In [1]:
from pathlib import Path
import pandas as pd

In [2]:
possible_paths = [


    Path('../data/labels/gold_set_Aya.csv')

]

gold_path = next(

    (path for path in possible_paths if path.exists()),

    None

)

if gold_path is None:

    raise FileNotFoundError(

        'Gold Set CSV not found. Check the project folder.'

    )

gold_df = pd.read_csv(gold_path)


In [3]:
print('Dataset shape:', gold_df.shape)

print('Column names:')

print(gold_df.columns.tolist())

print('First 5 rows:')

display(gold_df.head())

print('Missing values:')

display(gold_df.isnull().sum())

Dataset shape: (200, 8)
Column names:
['ad_id', 'title', 'brand', 'gold_model_key', 'review_notes', 'predicted_model_key', 'predicted_match_rule', 'confidence']
First 5 rows:


,ad_id,title,brand,gold_model_key,review_notes,predicted_model_key,predicted_match_rule,confidence
0,209004543,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple,apple iphone air,العنوان يذكر iPhone Air بالعربية؛ الموديل معرو...,unknown,NaN,NaN
1,209009284,iPhone 12 pro ايفون ١٢ برو,apple,apple iphone 12 pro,NaN,apple iphone 12 pro,apple:ordered_regex,NaN
2,208969053,آبل - آيفون 13,apple,apple iphone 13,NaN,apple iphone 13,apple:ordered_regex,NaN
3,208714508,iPhone 16 Pro Max,apple,apple iphone 16 pro max,NaN,apple iphone 16 pro max,apple:ordered_regex,NaN
4,208987623,آبل - آيفون 13 برو ماكس,apple,apple iphone 13 pro max,NaN,apple iphone 13 pro max,apple:ordered_regex,NaN


Missing values:


ad_id                     0
title                     0
brand                    10
gold_model_key            0
review_notes            175
predicted_model_key       0
predicted_match_rule     19
confidence              200
dtype: int64

In [4]:
gold = gold_df["gold_model_key"].astype("string").str.strip()
predicted = gold_df["predicted_model_key"].astype("string").str.strip()


valid = gold.notna() & predicted.notna()


correct = gold[valid] == predicted[valid]

total = len(correct)
correct_count = int(correct.sum())
wrong_count = int(total - correct_count)

accuracy = correct_count / total * 100 if total > 0 else 0

print("===== BASELINE RESULTS =====")
print("Total evaluated rows:", total)
print("Correct predictions:", correct_count)
print("Wrong predictions:", wrong_count)
print(f"Baseline matching accuracy: {accuracy:.2f}%")


===== BASELINE RESULTS =====
Total evaluated rows: 200
Correct predictions: 181
Wrong predictions: 19
Baseline matching accuracy: 90.50%


In [5]:

gold = gold_df["gold_model_key"].astype("string").str.strip()
predicted = gold_df["predicted_model_key"].astype("string").str.strip()

valid = gold.notna() & predicted.notna()
errors = gold_df.loc[valid & gold.ne(predicted)].copy()

print("Number of errors:", len(errors))

display(
    errors[
        [
            "ad_id",
            "title",
            "brand",
            "gold_model_key",
            "predicted_model_key",
            "predicted_match_rule",
            "confidence",
            "review_notes",
        ]
    ]
)


Number of errors: 19


,ad_id,title,brand,gold_model_key,predicted_model_key,predicted_match_rule,confidence,review_notes
0,209004543,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple,apple iphone air,unknown,NaN,NaN,العنوان يذكر iPhone Air بالعربية؛ الموديل معرو...
18,204168287,iPhone 15pro max 265,apple,apple iphone 15 pro max,apple iphone 15 pro,apple:ordered_regex,NaN,العنوان يذكر Pro Max؛ نتيجة القواعد أسقطت Max.
53,208966439,آبل - آيفون 4 اس,apple,apple iphone 4s,apple iphone 4,apple:ordered_regex,NaN,العنوان يذكر iPhone 4S؛ نتيجة القواعد أسقطت S.
60,207903641,Apple iPhone 16 Pro 128GB Desert Titanium,apple,apple iphone 16 pro,unknown,NaN,NaN,الموديل واضح من العنوان: iPhone 16 Pro؛ السعة ...
64,206362098,لو عندك ايفون 14 او 15 او 16 او 17 وعاوز تبيعو...,apple,unknown,apple iphone 14,apple:ordered_regex,NaN,إعلان عام يذكر عدة موديلات للاهتمام بالبيع/الت...
81,208753123,Apple - iPhone 15 Pro Max 100% شحنتين فقط,apple,apple iphone 15 pro max,apple iphone 15 pro,apple:ordered_regex,NaN,العنوان يذكر Pro Max؛ نتيجة القواعد أسقطت Max.
128,208965564,تليفون Poco x3 nfc 8/128,xiaomi,xiaomi poco x3 nfc,xiaomi poco x3,xiaomi:rule_08,NaN,احتفظت بـNFC لأنها جزء مميز في اسم Poco X3 NFC.
140,205483828,Poco x8 pro max,xiaomi,xiaomi poco x8 pro max,xiaomi poco x8 pro,xiaomi:rule_08,NaN,العنوان يذكر Pro Max؛ نتيجة القواعد أسقطت Max.
154,208705954,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,NaN,xiaomi 17 ultra,unknown,NaN,NaN,NaN
156,208965729,Turbo 4 pro كسر زيرو,NaN,xiaomi redmi turbo 4 pro,unknown,NaN,NaN,Turbo 4 Pro مفهوم كاسم Redmi؛ الماركة غير موجو...


In [6]:
from pathlib import Path


source_dirs = [
    Path("src/preprocessing"),
    Path("../src/preprocessing"),
]

source_dir = next(
    (p for p in source_dirs if p.exists()),
    None
)

if source_dir is None:
    print("Source folder not found. Current folder:", Path.cwd())
else:
    for filename in ["model_keys.py", "normalization.py"]:
        file_path = source_dir / filename

        print(f"\n{'=' * 20} {filename} {'=' * 20}")

        if file_path.exists():
            print(file_path.read_text(encoding="utf-8"))
        else:
            print(f"File not found: {file_path}")



==================== model_keys.py ====================
import re

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
BRAND_WORDS = {
    "شاومي": "xiaomi", "ريدمي": "redmi", "ردمي": "redmi", "بوكو": "poco",
    "اوبو": "oppo", "رينو": "reno", "فايند": "find",
    "ريلمي": "realme", "ريالمي": "realme", "نارزو": "narzo",
    "هونر": "honor", "هونور": "honor", "اونر": "honor", "ماجيك": "magic",
    "انفينكس": "infinix", "انفنكس": "infinix",
    "فيفو": "vivo", "هواوي": "huawei", "نوفا": "nova",
    "موتورولا": "motorola", "موتو": "moto", "ريزر": "razr",
    "ون بلس": "oneplus", "وان بلس": "oneplus", "ونبلس": "oneplus", "نورد": "nord",
    "تكنو": "tecno", "سبارك": "spark", "كامون": "camon", "فانتوم": "phantom", "بوفا": "pova",
    "ايتل": "itel", "جوجل": "google", "بيكسل": "pixel", "سوني": "sony",
    "اكسبيريا": "xperia", "اكسبريا": "xperia", "نوكيا": "nokia", "لينوفو": "lenovo",
    "انفنيكس": "infinix", "نوبيا": "nubia", "بورا": "pura", "ستايلس": "stylus",
    "رازر": "razr", 

In [7]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("\nFiles and folders here:")

for item in Path.cwd().iterdir():
    print(item.name)

Current directory: c:\Users\MG\Documents\DEPI-GRADUATION-PROJECT\notebooks

Files and folders here:
01_S0_split_and_baseline.ipynb
02_end_of_stage1_inspect_data.ipynb
03_Aya_stage2_error_analysis.ipynb
04_error_analysis_and_remeasure.ipynb
archive


In [8]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Does src exist?", (project_root / "src").exists())
print("Does model_keys.py exist?",
      (project_root / "src" / "preprocessing" / "model_keys.py").exists())

Project root: c:\Users\MG\Documents\DEPI-GRADUATION-PROJECT
Does src exist? True
Does model_keys.py exist? True


In [9]:
from src.preprocessing.model_keys import (
    extract_brand_model,
    extract_iphone_model,
)

print("Import successful!")

Import successful!


In [10]:
test_results = []

for _, row in errors.iterrows():
    title = str(row["title"])
    brand_value = row["brand"]

    # Skip rows with a missing brand for now
    if pd.isna(brand_value):
        continue

    brand = str(brand_value).strip().lower()

    if brand == "apple":
        prediction, rule = extract_iphone_model(
            title, with_rule=True
        )
    else:
        prediction, rule = extract_brand_model(
            title, brand, with_rule=True
        )

    gold = str(row["gold_model_key"]).strip()
    prediction_text = (
        str(prediction).strip()
        if prediction is not None
        else "unknown"
    )

    test_results.append({
        "ad_id": row["ad_id"],
        "title": title,
        "gold_model_key": gold,
        "old_prediction": row["predicted_model_key"],
        "test_prediction": prediction_text,
        "matched_rule": rule,
        "test_correct": prediction_text == gold,
    })

test_df = pd.DataFrame(test_results)

print("Errors tested:", len(test_df))
print("Correct predictions:", int(test_df["test_correct"].sum()))
print(
    "Accuracy on tested errors:",
    f"{test_df['test_correct'].mean() * 100:.2f}%"
)

display(test_df)

Errors tested: 14
Correct predictions: 9
Accuracy on tested errors: 64.29%


,ad_id,title,gold_model_key,old_prediction,test_prediction,matched_rule,test_correct
0,209004543,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,unknown,iphone air,apple:special_case_01,False
1,204168287,iPhone 15pro max 265,apple iphone 15 pro max,apple iphone 15 pro,iphone 15 pro max,apple:model_pattern_01,False
2,208966439,آبل - آيفون 4 اس,apple iphone 4s,apple iphone 4,iphone 4s,apple:special_case_02,False
3,207903641,Apple iPhone 16 Pro 128GB Desert Titanium,apple iphone 16 pro,unknown,iphone 16 pro,apple:model_pattern_06,False
4,206362098,لو عندك ايفون 14 او 15 او 16 او 17 وعاوز تبيعو...,unknown,apple iphone 14,unknown,apple:ambiguous_models,True
5,208753123,Apple - iPhone 15 Pro Max 100% شحنتين فقط,apple iphone 15 pro max,apple iphone 15 pro,iphone 15 pro max,apple:model_pattern_01,False
6,208965564,تليفون Poco x3 nfc 8/128,xiaomi poco x3 nfc,xiaomi poco x3,xiaomi poco x3 nfc,xiaomi:rule_08,True
7,205483828,Poco x8 pro max,xiaomi poco x8 pro max,xiaomi poco x8 pro,xiaomi poco x8 pro max,xiaomi:rule_08,True
8,208960096,realme c85 pro,realme c85 pro,realme c85,realme c85 pro,realme:rule_07,True
9,208956503,هونر اكس 9 سى سمارت,honor x9c smart,honor x9c,honor x9c smart,honor:rule_01,True


In [11]:
display(
    test_df[
        [
            "title",
            "gold_model_key",
            "old_prediction",
            "test_prediction",
            "matched_rule",
            "test_correct",
        ]
    ].sort_values("test_correct")
)

,title,gold_model_key,old_prediction,test_prediction,matched_rule,test_correct
0,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,unknown,iphone air,apple:special_case_01,False
1,iPhone 15pro max 265,apple iphone 15 pro max,apple iphone 15 pro,iphone 15 pro max,apple:model_pattern_01,False
2,آبل - آيفون 4 اس,apple iphone 4s,apple iphone 4,iphone 4s,apple:special_case_02,False
3,Apple iPhone 16 Pro 128GB Desert Titanium,apple iphone 16 pro,unknown,iphone 16 pro,apple:model_pattern_06,False
5,Apple - iPhone 15 Pro Max 100% شحنتين فقط,apple iphone 15 pro max,apple iphone 15 pro,iphone 15 pro max,apple:model_pattern_01,False
4,لو عندك ايفون 14 او 15 او 16 او 17 وعاوز تبيعو...,unknown,apple iphone 14,unknown,apple:ambiguous_models,True
6,تليفون Poco x3 nfc 8/128,xiaomi poco x3 nfc,xiaomi poco x3,xiaomi poco x3 nfc,xiaomi:rule_08,True
7,Poco x8 pro max,xiaomi poco x8 pro max,xiaomi poco x8 pro,xiaomi poco x8 pro max,xiaomi:rule_08,True
8,realme c85 pro,realme c85 pro,realme c85,realme c85 pro,realme:rule_07,True
9,هونر اكس 9 سى سمارت,honor x9c smart,honor x9c,honor x9c smart,honor:rule_01,True


In [12]:
import inspect

print("=== extract_iphone_model ===")
print(inspect.getsource(extract_iphone_model))

print("\n=== extract_brand_model ===")
print(inspect.getsource(extract_brand_model))

=== extract_iphone_model ===
def extract_iphone_model(title, with_rule=False):
    text = re.sub(r"\s+", " ", str(title).lower().translate(ARABIC_DIGITS)).strip()
    # Multiple alternative models in one title are not a single listing model.
    if re.search(
        r"(?:iphone|i\s*phone|ايفون|آيفون)\s*\d{1,2}.*?(?:\bor\b|او).*?\d{1,2}",
        text,
    ):
        return _prediction_result("unknown", "apple:ambiguous_models", with_rule)

    special_cases = [
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*(?:air|اير)\b", "iphone air"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*4\s*(?:s|اس|أس)\b", "iphone 4s"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*6\s*(?:s|اس|أس)\s*(?:plus|بلس)\b", "iphone 6s plus"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:s|اس|أس)\s*(?:max|ماكس)\b", "iphone xs max"),
        (r"(?:iphone|i\s*phone|ايفون|آيفون)\s*x\s*(?:max|ماكس)\b", "iphone x max"),
        (r"ip+h+o+n+e+\s*17\s+pro+m+a+x+", "iphone 17 pro max"),
        (r"ايفون+ن?\s*17\s*ب

In [13]:
from pathlib import Path

project_root = Path.cwd().parent
search_terms = [
    "extract_iphone_model(",
    "extract_brand_model(",
]

for file_path in (project_root / "src").rglob("*.py"):
    try:
        lines = file_path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        continue

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(term in line for term in search_terms)
    ]

    if matches:
        print(f"\n--- {file_path.relative_to(project_root)} ---")
        for line_number, line in matches:
            print(f"{line_number}: {line}")


--- src\preprocessing\model_keys.py ---
78: def extract_brand_model(title, brand, with_rule=False):
104: def extract_iphone_model(title, with_rule=False):
1824: model, match_rule = extract_iphone_model(title, with_rule=True)
1828: return extract_brand_model(title, brand, with_rule=with_rule)


In [14]:
from pathlib import Path

project_root = Path.cwd().parent
file_path = project_root / "src" / "preprocessing" / "model_keys.py"

lines = file_path.read_text(encoding="utf-8").splitlines()

for i in range(1795, min(1840, len(lines))):
    print(f"{i + 1}: {lines[i]}")

1796:         # LG G-series
1797:         (r"\bg\s*(\d{1,2})\b", "g{1}"),
1798: 
1799:         # LG V-series
1800:         (r"\bv\s*(\d{1,2})\b", "v{1}"),
1801: 
1802:         # LG K-series
1803:         (r"\bk\s*(\d{1,3})\b", "k{1}"),
1804: 
1805:         # LG Q-series
1806:         (r"\bq\s*(\d{1,3})\b", "q{1}"),
1807: 
1808:         # LG W-series
1809:         (r"\bw\s*(\d{1,3})\b", "w{1}"),
1810: 
1811:         # LG X-series
1812:         (r"\bx\s*(\d{1,3})\b", "x{1}"),
1813:     ],
1814: }
1815: 
1816: 
1817: def predict_model_key(title, brand, with_rule=False):
1818:     """Predict a canonical model key and, optionally, the rule that matched."""
1819:     brand = str(brand).strip().lower()
1820:     if not brand or brand == "nan":
1821:         return _prediction_result("unknown", "unknown_brand", with_rule)
1822: 
1823:     if brand == "apple":
1824:         model, match_rule = extract_iphone_model(title, with_rule=True)
1825:         model_key = "unknown" if model == "unknown" 

In [15]:
from src.preprocessing.model_keys import predict_model_key

test_results = []

for _, row in errors.iterrows():
    prediction, rule = predict_model_key(
        row["title"],
        row["brand"],
        with_rule=True
    )

    gold = str(row["gold_model_key"]).strip()

    test_results.append({
        "ad_id": row["ad_id"],
        "title": row["title"],
        "gold_model_key": gold,
        "old_prediction": row["predicted_model_key"],
        "test_prediction": prediction,
        "matched_rule": rule,
        "test_correct": str(prediction).strip() == gold,
    })

test_df = pd.DataFrame(test_results)

print("Errors tested:", len(test_df))
print("Correct predictions:", int(test_df["test_correct"].sum()))
print("Still incorrect:", int((~test_df["test_correct"]).sum()))

display(
    test_df[
        [
            "title",
            "gold_model_key",
            "old_prediction",
            "test_prediction",
            "matched_rule",
            "test_correct",
        ]
    ].sort_values("test_correct")
)

Errors tested: 19
Correct predictions: 14
Still incorrect: 5


,title,gold_model_key,old_prediction,test_prediction,matched_rule,test_correct
9,Turbo 4 pro كسر زيرو,xiaomi redmi turbo 4 pro,unknown,unknown,unknown_brand,False
12,Mate 30 pro 5g,huawei mate 30 pro 5g,unknown,unknown,unknown_brand,False
11,17 Pro Max,apple iphone 17 pro max,unknown,unknown,unknown_brand,False
10,لبيع ميت تن لايت,huawei mate 10 lite,unknown,unknown,unknown_brand,False
8,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,xiaomi 17 ultra,unknown,unknown,unknown_brand,False
16,vivo v40 ite,vivo v40 lite,vivo v40,vivo v40 lite,vivo:rule_05,True
15,Vivo V60 lite 5g,vivo v60 lite 5g,vivo v60 lite,vivo v60 lite 5g,vivo:rule_05,True
14,هونر اكس 9 سى سمارت,honor x9c smart,honor x9c,honor x9c smart,honor:rule_01,True
13,realme c85 pro,realme c85 pro,realme c85,realme c85 pro,realme:rule_07,True
0,ايفون اير نسخه الالمانيه مدفوع الضريبه ماشا ال...,apple iphone air,unknown,apple iphone air,apple:special_case_01,True


In [16]:
display(
    test_df.loc[
        ~test_df["test_correct"],
        [
            "ad_id",
            "title",
            "gold_model_key",
            "old_prediction",
            "test_prediction",
            "matched_rule",
        ]
    ]
)

,ad_id,title,gold_model_key,old_prediction,test_prediction,matched_rule
8,208705954,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,xiaomi 17 ultra,unknown,unknown,unknown_brand
9,208965729,Turbo 4 pro كسر زيرو,xiaomi redmi turbo 4 pro,unknown,unknown,unknown_brand
10,206721198,لبيع ميت تن لايت,huawei mate 10 lite,unknown,unknown,unknown_brand
11,208954753,17 Pro Max,apple iphone 17 pro max,unknown,unknown,unknown_brand
12,208989408,Mate 30 pro 5g,huawei mate 30 pro 5g,unknown,unknown,unknown_brand


In [17]:
from pathlib import Path

project_root = Path.cwd().parent

for file_path in (project_root / "src").rglob("*.py"):
    try:
        lines = file_path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        continue

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(word in line.lower() for word in [
            "detect_brand",
            "infer_brand",
            "brand_from",
            "extract_brand",
            "predict_model_key(",
        ])
    ]

    if matches:
        print(f"\n--- {file_path.relative_to(project_root)} ---")
        for line_number, line in matches:
            print(f"{line_number}: {line}")


--- src\preprocessing\model_keys.py ---
78: def extract_brand_model(title, brand, with_rule=False):
1817: def predict_model_key(title, brand, with_rule=False):
1828: return extract_brand_model(title, brand, with_rule=with_rule)


In [18]:
from src.preprocessing.model_keys import BRAND_CONFIG

print("Supported brands:")
for brand in sorted(BRAND_CONFIG.keys()):
    print("-", brand)

Supported brands:
- google
- honor
- huawei
- infinix
- itel
- lenovo
- lg
- motorola
- nokia
- oneplus
- oppo
- realme
- samsung
- sony
- tecno
- vivo
- xiaomi
- zte


In [19]:
missing_brand_errors = errors.loc[errors["brand"].isna()].copy()

def infer_brand_from_title(title):
    text = str(title).lower()

    if any(x in text for x in ["xiaomi", "mi ", "poco", "redmi", "شاومي", "ريدمي"]):
        return "xiaomi"

    if any(x in text for x in ["huawei", "هواوي", "ميت", "mate"]):
        return "huawei"

    if any(x in text for x in ["iphone", "ايفون", "آيفون", "apple", "آبل"]):
        return "apple"

    if "google pixel" in text or "pixel" in text:
        return "google"

    return None

missing_brand_errors["inferred_brand"] = (
    missing_brand_errors["title"].apply(infer_brand_from_title)
)

display(
    missing_brand_errors[
        ["title", "brand", "inferred_brand", "gold_model_key"]
    ]
)

,title,brand,inferred_brand,gold_model_key
154,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,NaN,xiaomi,xiaomi 17 ultra
156,Turbo 4 pro كسر زيرو,NaN,NaN,xiaomi redmi turbo 4 pro
157,لبيع ميت تن لايت,NaN,huawei,huawei mate 10 lite
159,17 Pro Max,NaN,NaN,apple iphone 17 pro max
160,Mate 30 pro 5g,NaN,huawei,huawei mate 30 pro 5g


In [20]:
from src.preprocessing.model_keys import predict_model_key

def infer_brand_from_title(title):
    text = str(title).lower()

    if any(x in text for x in [
        "xiaomi", "mi ", "poco", "redmi", "شاومي", "ريدمي"
    ]) or "turbo" in text:
        return "xiaomi"

    if any(x in text for x in [
        "huawei", "هواوي", "mate", "ميت"
    ]):
        return "huawei"

    if any(x in text for x in [
        "iphone", "ايفون", "آيفون", "apple", "آبل"
    ]):
        return "apple"

    if "pixel" in text:
        return "google"

    return None


results = []

for _, row in missing_brand_errors.iterrows():
    inferred_brand = infer_brand_from_title(row["title"])

    if inferred_brand is None:
        prediction, rule = "unknown", "unknown_brand_not_inferred"
    else:
        prediction, rule = predict_model_key(
            row["title"], inferred_brand, with_rule=True
        )

    results.append({
        "title": row["title"],
        "inferred_brand": inferred_brand,
        "gold_model_key": row["gold_model_key"],
        "test_prediction": prediction,
        "matched_rule": rule,
        "correct": str(prediction).strip() == str(row["gold_model_key"]).strip(),
    })

missing_brand_test = pd.DataFrame(results)

display(missing_brand_test)

,title,inferred_brand,gold_model_key,test_prediction,matched_rule,correct
0,Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم...,xiaomi,xiaomi 17 ultra,xiaomi mi 17 ultra,xiaomi:rule_14,False
1,Turbo 4 pro كسر زيرو,xiaomi,xiaomi redmi turbo 4 pro,unknown,xiaomi:no_match,False
2,لبيع ميت تن لايت,huawei,huawei mate 10 lite,unknown,huawei:no_match,False
3,17 Pro Max,NaN,apple iphone 17 pro max,unknown,unknown_brand_not_inferred,False
4,Mate 30 pro 5g,huawei,huawei mate 30 pro 5g,huawei mate 30 pro,huawei:rule_07,False


In [21]:
from src.preprocessing.model_keys import BRAND_CONFIG

for brand in ["xiaomi", "huawei"]:
    print(f"\n===== {brand.upper()} RULES =====")

    for i, (pattern, template) in enumerate(
        BRAND_CONFIG[brand]["rules"], start=1
    ):
        print(f"{i:02d}. Pattern: {pattern}")
        print(f"    Template: {template}")


===== XIAOMI RULES =====
01. Pattern: \bblack\s*shark\s*(\d{1,2})(?:\s*(pro))?\b
    Template: black shark {1} {2}
02. Pattern: \bredmi\s*note\s*(\d{1,2})\s*(s|t|a)?\s*(pro\s*plus|pro\s*max|pro|plus)?\s*(5g)?\b
    Template: redmi note {1}{2} {3} {4}
03. Pattern: \bredmi\s*k\s*(\d{2,3})\s*(pro\s*max|pro|ultra|plus)?\b
    Template: redmi k{1} {2}
04. Pattern: \bredmi\s*turbo\s*(\d{1,2})\s*(pro|max)?\b
    Template: redmi turbo {1} {2}
05. Pattern: \bredmi\s*([ac])\s*(\d{1,2})(y)?\b
    Template: redmi {1}{2}{3}
06. Pattern: \bredmi\s*(\d{1,2})\s*(c|a|t|s|x)\s*(prime|pro|plus)?\b
    Template: redmi {1}{2} {3}
07. Pattern: \bredmi\s*(\d{1,2})\s*(prime|pro|plus)?\b
    Template: redmi {1} {2}
08. Pattern: \bpoco\s*([xfmc])\s*(\d{1,2})\s*(pro\s*max|nfc|pro|max|gt|ultra|plus)?\b
    Template: poco {1}{2} {3}
09. Pattern: \bpoco\b.*?\b([xfmc])\s*(\d{1,2})\s*(pro\s*max|nfc|pro|max|gt|ultra|plus)?\b
    Template: poco {1}{2} {3}
10. Pattern: \bmix\s*(fold|flip)\s*(\d{1,2})?\b
    Template: m

In [22]:
import re
from src.preprocessing.model_keys import predict_model_key

def experimental_prediction(title, brand):
    text = str(title).lower()

    # Experimental Xiaomi rules
    if brand == "xiaomi":
        mi_match = re.search(
            r"\bmi\s*(\d{1,2})\s*(pro|lite|ultra)?\b",
            text
        )
        if mi_match:
            number = mi_match.group(1)
            suffix = mi_match.group(2) or ""
            return f"xiaomi {number} {suffix}".strip(), "test:mi_model"

        turbo_match = re.search(
            r"\bturbo\s*(\d{1,2})\s*(pro|max)?\b",
            text
        )
        if turbo_match:
            number = turbo_match.group(1)
            suffix = turbo_match.group(2) or ""
            return f"xiaomi redmi turbo {number} {suffix}".strip(), "test:turbo_model"

    # Experimental Huawei 5G rule
    if brand == "huawei":
        mate_match = re.search(
            r"\bmate\s*(\d{1,2})\s*(pro|lite|plus)?\s*(5g)?\b",
            text
        )
        if mate_match:
            number = mate_match.group(1)
            suffix = mate_match.group(2) or ""
            network = mate_match.group(3) or ""
            return f"huawei mate {number} {suffix} {network}".strip(), "test:mate_model"

    return predict_model_key(title, brand, with_rule=True)


for _, row in missing_brand_errors.iterrows():
    brand = infer_brand_from_title(row["title"])

    if brand is None:
        prediction, rule = "unknown", "unknown_brand_not_inferred"
    else:
        prediction, rule = experimental_prediction(row["title"], brand)

    print("\nTitle:", row["title"])
    print("Gold: ", row["gold_model_key"])
    print("Test: ", prediction)
    print("Rule: ", rule)


Title: Mi 17 Ultra dual sim 512/1TB Black جديد متبرشم بضمان الوكيل
Gold:  xiaomi 17 ultra
Test:  xiaomi 17 ultra
Rule:  test:mi_model

Title: Turbo 4 pro كسر زيرو
Gold:  xiaomi redmi turbo 4 pro
Test:  xiaomi redmi turbo 4 pro
Rule:  test:turbo_model

Title: لبيع ميت تن لايت
Gold:  huawei mate 10 lite
Test:  unknown
Rule:  huawei:no_match

Title: 17 Pro Max
Gold:  apple iphone 17 pro max
Test:  unknown
Rule:  unknown_brand_not_inferred

Title: Mate 30 pro 5g
Gold:  huawei mate 30 pro 5g
Test:  huawei mate 30 pro 5g
Rule:  test:mate_model


In [23]:
from src.preprocessing.model_keys import (
    prepare_text,
    BRAND_CONFIG,
)

title = "لبيع ميت تن لايت"
cfg = BRAND_CONFIG["huawei"]

print("Original title:")
print(title)

print("\nNormalized title:")
print(
    prepare_text(
        title.lower(),
        cfg.get("letters"),
        cfg.get("aliases"),
        cfg.get("pre"),
    )
)

print("\nHuawei aliases:")
print(cfg.get("aliases"))

print("\nHuawei preprocessing rules:")
print(cfg.get("pre"))

Original title:
لبيع ميت تن لايت

Normalized title:
ل p ع mate تن lite

Huawei aliases:
{'هواوي': 'huawei', 'هواوى': 'huawei', 'هواويه': 'huawei', 'ميت': 'mate', 'نوفا': 'nova', 'بورا': 'pura', 'برايم': 'prime', 'برو': 'pro', 'لايت': 'lite', 'بلس': 'plus', 'الترا': 'ultra', 'التميت': 'ultimate', 'بي': 'p', 'بى': 'p', 'واي': 'y', 'واى': 'y', 'سي': 'se'}

Huawei preprocessing rules:
[('(?<![\\u0600-\\u06FF])اكس(?![\\u0600-\\u06FF])', 'x'), ('ميت\\s*(?=[a-z0-9])', 'mate '), ('نوفا\\s*(?=\\d)', 'nova '), ('بورا\\s*(?=\\d)', 'pura '), ('\\bmat(?=\\s*\\d)', 'mate '), ('\\bhuawei(?=[a-z])', 'huawei '), ('\\b(y|p|nova|mate|pura)\\s*(?=\\d)', '\\1 '), ('\\bmate\\s*x\\s*t\\s*(?=\\d)', 'mate xt '), ('\\bmate\\s*x\\s*(?=\\d)', 'mate x '), ('\\by\\s*(\\d{1,2})([a-z])\\b', 'y \\1\\2'), ('\\by\\s*(9)\\s*(2019)\\b', 'y \\1 \\2'), ('(?<=\\d)(pro|max|plus|ultra|lite|prime|ultimate)\\b', ' \\1')]


In [24]:
import re

title = "لبيع ميت تن لايت"

normalized = prepare_text(
    title.lower(),
    cfg.get("letters"),
    cfg.get("aliases"),
    cfg.get("pre"),
)

# Experimental rule: Arabic "تن" represents model number 10 here
test_text = re.sub(r"\bmate\s+تن\s+lite\b", "mate 10 lite", normalized)

print("Before:", normalized)
print("After: ", test_text)

Before: ل p ع mate تن lite
After:  ل p ع mate 10 lite


In [25]:
prediction, rule = predict_model_key(
    test_text,
    "huawei",
    with_rule=True
)

print("Prediction:", prediction)
print("Gold:      ", "huawei mate 10 lite")
print("Rule:      ", rule)
print("Correct:   ", prediction == "huawei mate 10 lite")

Prediction: huawei mate 10 lite
Gold:       huawei mate 10 lite
Rule:       huawei:rule_07
Correct:    True


In [26]:
results = []

for _, row in gold_df.iterrows():
    prediction, rule = predict_model_key(
        row["title"],
        row["brand"],
        with_rule=True
    )

    results.append({
        "ad_id": row["ad_id"],
        "gold": row["gold_model_key"],
        "prediction": prediction,
        "rule": rule,
        "correct": prediction == row["gold_model_key"]
    })

evaluation_df = pd.DataFrame(results)

total = len(evaluation_df)
correct = evaluation_df["correct"].sum()
accuracy = correct / total * 100

print(f"Total evaluated: {total}")
print(f"Correct: {correct}")
print(f"Incorrect: {total - correct}")
print(f"Accuracy: {accuracy:.2f}%")

Total evaluated: 200
Correct: 195
Incorrect: 5
Accuracy: 97.50%


In [27]:
remaining_errors = evaluation_df.loc[
    ~evaluation_df["correct"],
    ["ad_id", "gold", "prediction", "rule"]
]

display(remaining_errors)

,ad_id,gold,prediction,rule
154,208705954,xiaomi 17 ultra,unknown,unknown_brand
156,208965729,xiaomi redmi turbo 4 pro,unknown,unknown_brand
157,206721198,huawei mate 10 lite,unknown,unknown_brand
159,208954753,apple iphone 17 pro max,unknown,unknown_brand
160,208989408,huawei mate 30 pro 5g,unknown,unknown_brand
